# 03 · Coordenação e teste preditivo
Etapas 03.5–03.6. Os testes estão pré-definidos em `docs/plano_de_testes.md` — **não** acrescente especificações depois de ver resultados sem registrar em `docs/decisoes.md`.

In [ ]:
import sys; sys.path.insert(0, '..')
import pandas as pd, matplotlib.pyplot as plt
from src import config, limpeza
plt.rcParams.update({'figure.figsize': (8, 3.5), 'axes.spines.top': False, 'axes.spines.right': False})
from src import coordenacao as co, preditivo as pr
cl = pd.read_parquet(config.DATA_PROC/'clusters_todos.parquet')
rev = pd.read_csv(config.CLUSTERS_REVISADOS_CSV)
print(len(cl), 'clusters detectados;', len(rev), 'revisados')
cl[['cluster_id','tipo','ticker','n_posts','n_autores','span_min','escore']].head(10)

## Revisão humana dos clusters

In [ ]:
print(co.validar_revisao(rev))
co.precisao_no_topo(rev)

In [ ]:
rev.groupby('decisao_humana')['escore'].describe()[['count','mean','min','max']]

## Teste preditivo
Execute `python -m src.run_pipeline preditivo` (gera `reports/resultados_preditivo.csv`).

In [ ]:
tab = pd.read_csv(config.REPORTS/'resultados_preditivo.csv')
tab[tab.modelo=='controles'][['versao','dv','coef','ep','p','p_holm','n','n_tickers']].round(4)

## Coeficiente do índice (modelo com controles) — IC 95%

In [ ]:
t = tab[(tab.modelo=='controles') & tab.coef.notna()].copy()
t['lbl'] = t['versao'] + ' · ' + t['dv']
plt.figure(figsize=(8, 0.45*len(t)+1))
plt.errorbar(t['coef'], range(len(t)), xerr=1.96*t['ep'], fmt='o')
plt.yticks(range(len(t)), t['lbl']); plt.axvline(0, color='k', lw=.8)
plt.title('Efeito do índice de sentimento (IC 95%)'); plt.tight_layout(); plt.show()

## Leitura honesta
- O placebo (`ret_lag1`) é significativo? Se sim, sinal de causalidade invertida.
- O sinal sobrevive ao controle `ret_t` e à correção de Holm?
- Quantas observações (n) e quantos tickers? Poder estatístico?
- Resultado nulo bem demonstrado é um resultado válido.

In [ ]:
pr.conclusao(tab.assign(familia_principal=tab['familia_principal']))